# 01c — Synthetic support tickets EDA

Quick profile of the synthetic Contoso / M365 support-tickets dataset generated by `src/synth_tickets/`.

The dataset is meant to be **closer to Contoso's product domain** than `publicholidays` or `nyctaxi_yellow`, so partners can immediately re-skin the same training + deploy pipeline.

Two label columns make this dataset useful for **two parallel models**:

- `priority_actual` — multi-class classification (Low / Medium / High / Critical)
- `sla_breached` — binary classification (will this ticket miss SLA?)

Run the loader first to (re)create the table in OneLake:

```bash
python -m src.load_synth_tickets_to_fabric --rows 200000 --table support_tickets
```

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_settings
from src.data import read_delta_table

TABLE = 'support_tickets'

## 1. Load the table

Same `read_delta_table` helper used everywhere else in the repo. Falls back to the local snapshot if the OneLake table doesn't exist yet (handy when iterating without re-uploading).

In [ ]:
try:
    df = read_delta_table(load_settings(), table=TABLE)
    src = 'OneLake'
except Exception as exc:
    print(f'OneLake read failed ({exc.__class__.__name__}); falling back to local snapshot.')
    df = pd.read_parquet('../data/local/support_tickets.parquet')
    src = 'local snapshot'

print(f'Loaded {len(df):,} rows from {src}')
df.info()

## 2. Schema + sample rows

In [ ]:
df.head(10)

## 3. Label distributions

We expect realistic class imbalance: most tickets are Low/Medium, fewer are Critical. SLA breach rate should sit in the 25–40% range for a believable 'still-improving support org' story.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

priority_counts = df['priority_actual'].value_counts().reindex(['Low', 'Medium', 'High', 'Critical'])
priority_counts.plot.bar(ax=axes[0], color='#4C78A8')
axes[0].set_title('priority_actual')
axes[0].set_ylabel('tickets')

breach_counts = df['sla_breached'].value_counts()
breach_counts.plot.bar(ax=axes[1], color=['#54A24B', '#E45756'])
axes[1].set_title(f"sla_breached  ({df['sla_breached'].mean():.1%} breach rate)")
axes[1].set_xticklabels(['Met SLA', 'Breached'], rotation=0)

plt.tight_layout()
plt.show()

## 4. Signal check — do the features actually move the labels?

If labels were random, the bars below would all be the same height. We expect Enterprise tenants and `Data loss` / `Compliance` / `Auth` categories to skew Critical, validating that there's learnable signal.

In [ ]:
def stacked_priority(col, ax):
    pct = (
        df.groupby(col)['priority_actual']
        .value_counts(normalize=True)
        .unstack(fill_value=0)
        .reindex(columns=['Low', 'Medium', 'High', 'Critical'])
    )
    pct.plot.bar(stacked=True, ax=ax, colormap='RdYlGn_r')
    ax.set_title(f'priority_actual mix by {col}')
    ax.set_ylabel('share')
    ax.legend(loc='upper right', fontsize=7)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
stacked_priority('customer_tier', axes[0])
stacked_priority('issue_category', axes[1])
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
breach_by_priority = df.groupby('priority_actual')['sla_breached'].mean().reindex(
    ['Low', 'Medium', 'High', 'Critical']
)
breach_by_priority.plot.bar(ax=ax, color='#E45756')
ax.set_title('SLA breach rate by priority_actual')
ax.set_ylabel('breach rate')
ax.set_ylim(0, 1)
for i, v in enumerate(breach_by_priority):
    ax.text(i, v + 0.02, f'{v:.0%}', ha='center')
plt.tight_layout()
plt.show()

## 5. Tenant volume long tail

Want to see a long-tail distribution: a few big tenants generate a lot of the volume, lots of tenants file occasional tickets. Mirrors real SaaS support patterns.

In [ ]:
tenant_volume = df['tenant_id'].value_counts()
print(f'Unique tenants: {tenant_volume.shape[0]}')
print(f'Top 20 tenants drive {tenant_volume.head(20).sum() / len(df):.1%} of all tickets')

fig, ax = plt.subplots(figsize=(9, 3))
tenant_volume.head(50).plot.bar(ax=ax, color='#4C78A8')
ax.set_title('Top 50 tenants by ticket volume')
ax.set_ylabel('tickets')
ax.set_xticklabels([])
plt.tight_layout()
plt.show()

## 6. Next steps

The dataset is registered. To train a model on it, mirror the existing publicholidays flow:

```bash
# Multi-class priority model:
python -m src.train \
    --data-path data/local/support_tickets.parquet \
    --target priority_actual \
    --drop-cols ticket_id created_at tenant_id sla_breached \
    --output-dir outputs/tickets-priority-model

# Or for the binary SLA model:
python -m src.train \
    --data-path data/local/support_tickets.parquet \
    --target sla_breached \
    --drop-cols ticket_id created_at tenant_id priority_actual \
    --output-dir outputs/tickets-sla-model
```

Then submit as an AML job with `notebooks/02-sklearn-training.ipynb` (swap `DATA_ASSET` + `TARGET` + `MODEL_NAME`), register the model, and deploy via `src/deploy_online_endpoint.py` to a new endpoint. Same helpers, different dataset — that's the partner story.